In [ ]:
!pip install -U diffusers transformers accelerate -q

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, requests
import matplotlib.pyplot as plt
from PIL import Image
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from diffusers import StableDiffusion3Pipeline

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

print(f"GPU: {torch.cuda.get_device_name(0)}")

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    text_encoder_3=None,
    tokenizer_3=None,
    dtype=torch.float16,
    device_map="cuda",   # يحمّل مباشرة على GPU بصيغة float16 بخطوة واحدة، بدل from_pretrained ثم .to() منفصلة
)

print("✅ الموديل جاهز")

In [ ]:
def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, professional corporate branding, flat vector design, Adobe Illustrator style, geometric, minimalist icon, clean background, no text, no cartoon, no mascot"

briefs = requests.get("https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json").json()[:5]

generated_files = []
for brief in briefs:
    print(f"⏳ {brief['id']}...")
    image = pipe(
        prompt=brief_to_image_prompt(brief),
        num_inference_steps=28,
        guidance_scale=7.0,
        generator=torch.Generator("cuda").manual_seed(42),
    ).images[0]

    filename = f"SD3_{brief['id']}.png"
    image.save(filename)
    generated_files.append(filename)
    print(f"✅ {brief['id']}")

    del image
    gc.collect()
    torch.cuda.empty_cache()

print("✅ خلصت")